In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir('/zhome/71/c/146676/main/')
import SimpleITK as sitk
from loaders import loader_XA_to_NA
import importlib
importlib.reload(loader_XA_to_NA)
import tifffile
import SimpleITK
from loaders import stitcher_XA
from helpers import module_auxiliary as ma
from PIL import Image
import imageio.v3 as iio
Image.MAX_IMAGE_PIXELS = None 
from PIL import Image
import IPython.display as display
from scipy.ndimage import gaussian_filter
from scipy.ndimage import label, find_objects
from matplotlib_scalebar.scalebar import ScaleBar
from cil.framework import ImageGeometry, ImageData # For extract edge function
from cil.optimisation.operators import GradientOperator # For extract edge function
import matplotlib.colors as mcolors
%matplotlib inline

In [ ]:
# 15 distinct, bright RGB colors
import matplotlib.colors as mcolors
colors = [
    (255, 0, 0),      # Red
    (0, 255, 0),      # Green
    (0, 0, 255),      # Blue
    (255, 255, 0),    # Yellow
    (255, 165, 0),    # Orange
    (255, 0, 255),    # Magenta
    (0, 255, 255),    # Cyan
    (128, 0, 128),    # Purple
    (0, 128, 128),    # Teal
    (128, 128, 0),    # Olive
    (255, 105, 180),  # Pink
    (139, 69, 19),    # Brown
    (75, 0, 130),     # Indigo
    (0, 255, 127),     # Spring Green
    (127, 127, 127)   # Grey
]

# Convert to [0,1] scale for matplotlib
colors = [(r/255, g/255, b/255) for r, g, b in colors]
# Generate colormaps that transition from black to each color
colormaps = []
for color in colors:
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", [(0, 0, 0), color])
    colormaps.append(cmap)

In [ ]:
path = '/dtu-compute/msaca/sliceA_eds/EDS_BB_A_raw_files/'
pre = 'Mosaic element_'
pre = 'eds'
images = []
names = [pre + 'Al.tiff', pre + 'Ca.tiff',pre + 'Cl.tiff', pre + 'Cr.tiff', pre + 'Fe.tiff',
                    pre + 'K.tiff', pre + 'Mg.tiff', pre + 'Na.tiff', pre + 'Ni.tiff',
                    pre + 'O.tiff', pre + 'P.tiff', pre + 'S.tiff', pre + 'Si.tiff',
                    pre + 'Ti.tiff', pre + 'Zr.tiff']
for i in range(len(names)):
    images.append(tifffile.imread(path + names[i]))
images = np.stack(images)
images = np.stack(images)
images[images>150]=0
images[0,images[0]>15]=0
images_d = images[:,::2,::2]
n_chan, ny, nx = np.shape(images_d)
factor = np.linspace(10,30,nx)
images_d[0] = images_d[0]*factor[np.newaxis,:]
factor = np.linspace(1,1.5,nx)
images_d[1:] = images_d[1:]*factor[np.newaxis, np.newaxis,:]
print(np.shape(images))

In [ ]:
data_diffused = np.load('/dtu-compute/msaca/cache/eds_diffusioned.npy')
data_diffused = np.load('/dtu-compute/msaca/cache/eds_diffusioned_200.npy')

In [ ]:
N = 1
plt.imshow(data_diffused[N,1000:4000,0000:3000])
plt.clim([0,30])
plt.show()
plt.imshow(images_d[N,1000:4000,000:3000])
plt.clim([0,30])
plt.show()

plt.imshow(images_d[N,1000:4000,0000:3000]-data_diffused[N,1000:4000,0000:3000])
plt.clim([0,30])
plt.clim([0,5])
plt.show()
image_fil_  = sitk.GetImageFromArray(data_diffused[N])
image_fil = data_diffused[N]

In [ ]:
def xi_vector_field(image_s,eta,weigths):
        nc, ny, nx = np.shape(image_s)
        out_grad = np.zeros((ny,nx))
        for i in range(nc):
                print('Channel', i)
                ig = ImageGeometry(voxel_num_x=nx, voxel_num_y=ny)

                image_ = ImageData(image_s[i].astype(np.float32), geometry=ig)
                G = GradientOperator(ig)
                numerator_ = G.direct(image_)
                denominator_ = np.sqrt(eta**2 + numerator_.get_item(0)**2 + numerator_.get_item(1)**2)
                xi_ = numerator_/denominator_
                dy = xi_.get_item(0).as_array()
                dx = xi_.get_item(1).as_array()
                out_grad = out_grad + (dy**2 + dx**2)*weights[i]**2

        return np.sqrt(out_grad)/(np.sum(weights))


In [ ]:
etas = [10]
weights = np.ones(len(names))
for eta in etas:
    edges = xi_vector_field(data_diffused,eta,weights)
    edges_ = sitk.GetImageFromArray(edges)
    plt.imshow((edges)[2000:4000,1000:3000])
    plt.clim([0.000,0.03])
    plt.show()


In [ ]:
    plt.imshow((edges)[2000:4000,1000:3000])
    plt.clim([0.000,0.01])

In [ ]:
edges_subset = edges[2000:4000,1000:3000]
edges_subset_ = sitk.GetImageFromArray(edges_subset)
watershed_filter = sitk.MorphologicalWatershedImageFilter()
watershed_filter.SetMarkWatershedLine(True)#watershed_line)  # Prevent marking edges as lines
watershed_filter.SetFullyConnected(False)  # Use fully connected components for better segmentation
watershed_filter.SetLevel(0.005)

# Perform watershed segmentation using seeds
watershed_classes_ = watershed_filter.Execute(edges_subset_)

In [ ]:
watershed_classes = sitk.GetArrayFromImage(watershed_classes_)
print(np.max(watershed_classes))
plt.imshow(watershed_classes)
plt.show()

In [ ]:
def keep_largest_blobs(mask, num_blobs):
    # Label connected components
    labeled_mask, num_features = label(mask)
    
    # Get sizes of blobs
    blob_sizes = np.bincount(labeled_mask.ravel())[1:]  # Exclude background (label 0)
    
    if num_features == 0:
        return np.zeros_like(mask)  # No blobs found
    
    # Sort blobs by size (descending order) and get top N
    largest_blobs = np.argsort(blob_sizes)[::-1][:num_blobs] + 1  # +1 to match label indices
    
    # Create output mask with only the selected blobs
    output_mask = np.isin(labeled_mask, largest_blobs).astype(bool)
    
    return output_mask

In [ ]:
from segmentation import s3_watershed_eds
importlib.reload(s3_watershed_eds)
full_seg, segments, elem_segm = s3_watershed_eds.get_segmentation_from_watersheds(thresholds = np.array([15, 15, 15, 10 , 20, 10, 30, 12, 5 , 30, 10, 13 , 92, 10, 5]),
    output=True)

In [ ]:
# Add black as the first color
colors_d = [
    (0, 0, 255),      # Blue Plagio
    (128, 0, 128),    # Purple Alk
    (255, 50, 150),   # Pink   #cpx
    (0, 255, 0),      # Green   # opx
    (255, 165, 0),    # Orange     #apatite
    (255, 255, 0),    # Yellow Ilminite
    (0, 170, 255),     # light blue, chromite
    (255, 0, 0),    #  pyrite   
    (255, 0, 255),   # Magenta, Baddelyite
]
#colors = [(0, 0, 0)] + colors
# Create a ListedColormap
colors_d = [(0, 0, 0)] + [(r/255, g/255, b/255) for r, g, b in colors_d]
cmap_d = mcolors.ListedColormap(colors_d)
boundaries = np.arange(-0.5, 10, 1)  # [-0.5, 0.5, 1.5, ..., 14.5]
norm = mcolors.BoundaryNorm(boundaries, cmap_d.N)
# Plot the matrix with the custom colormap

In [ ]:
plt.close('all')
fig, ax = plt.subplots(figsize=(10, 10))

im = ax.imshow(full_seg, cmap=cmap_d, norm=norm,  alpha=0.7)

# Create a colorbar next to the plot
cbar = fig.colorbar(im, ax=ax, ticks=[], fraction=0.05, pad=0.04)
bounds = np.arange(len(colors_d) + 1)  # Define color boundaries
# Manually add text labels next to the colorbar
class_labels=['Background', 'Feldspar an-al', 'Feldspar al-or', 'Pyroxene (Cpx)', 'Pyroxene (Opx)', 'Apatite', 'Ilminite', 'Chromite', 'Pyrite', 'Baddelyite']
cbar_ax = cbar.ax  # Get the colorbar axis
for i, label in enumerate(class_labels):
    y_pos = (bounds[i] + bounds[i + 1]-1) / 2  # Center text at each color
    cbar_ax.text(1.3, y_pos, label, va='center', ha='left', fontsize=12)

cbar_ax.set_frame_on(False)  # Remove border

# Overlay the grayscale image using transparency
ax.imshow(images_d[6], cmap="gray", alpha=0.5, vmin = 0, vmax = 100)  # Adjust alpha for blending

ax.axis('off')
# Add scale bar (adjust values based on real-world scale)
scalebar = ScaleBar(1.82, "µm", location="lower right", color="white", scale_loc="bottom", box_alpha=0.5)
ax.add_artist(scalebar)
plt.tight_layout()
plt.savefig('plots/output_segm_full.png', bbox_inches='tight', dpi = 200)
plt.show()